In [1]:
import os
import pandas as pd
from dotenv import load_dotenv


# 載入環境變數中的 Token
load_dotenv()
token = os.getenv("FINMIND_TOKEN")

def get_financial_data(stock_id, start_date):
    """
    具備 Cache 機制的資料抓取函數
    邏輯：優先檢查 data/ 資料夾，若無才呼叫 API
    """
    # 建立存檔路徑 (例如: data/2330_financial_statement.csv)
    file_path = f"data/{stock_id}_financial.csv"
    

    # --- 1. 檢查快取 (DSA 邏輯：減少重複運算/請求) ---
    if os.path.exists(file_path):
        print(f"找到本地快取：{file_path}，直接讀取...")
        return pd.read_csv(file_path)

    # --- 2. 呼叫 API ---
    print(f"正在從 FinMind 抓取 {stock_id} 的財報資料...")
    dl = DataLoader()

    
    # 抓取綜合損益表
    df = dl.taiwan_stock_financial_statement(
        stock_id=stock_id,
        start_date=start_date
    )

    # --- 3. 儲存快取 (Persistence) ---
    if not df.empty:
        df.to_csv(file_path, index=False)
        print(f"資料抓取成功並已存至 {file_path}")
    else:
        print("警告：API 回傳空資料，請檢查 Token 或股票代碼。")
        
    return df

# --- 執行測試 ---
# 試試看抓取台積電 (2330)
df_2330 = get_financial_data("2330", "2023-01-01")
df_2330.head()


找到本地快取：data/2330_financial.csv，直接讀取...


,date,stock_id,type,value,origin_name
0,2023-03-31,2330,OperatingExpenses,5.530934e+10,營業費用
1,2023-03-31,2330,TAX,3.732590e+10,所得稅費用（利益）
2,2023-03-31,2330,EPS,7.980000e+00,基本每股盈餘（元）
3,2023-03-31,2330,PreTaxIncome,2.442749e+11,稅前淨利（淨損）
4,2023-03-31,2330,IncomeFromContinuingOperations,2.069490e+11,繼續營業單位本期淨利（淨損）


In [2]:
# =============================================================================
# CacheManager：快取管理器
# =============================================================================
# 功能：檢查、讀取、寫入快取，支援不同資料類型的過期策略
# 對標 C++：類似封裝好的 std::map + std::filesystem 操作
# =============================================================================

import pandas as pd
from pathlib import Path
from datetime import datetime

class CacheManager:
    """
    快取管理器
    
    職責：
    1. 生成快取檔案路徑 (依據 stock_id, data_type, 日期範圍)
    2. 檢查快取是否存在且有效 (未過期)
    3. 讀取/寫入快取
    """
    
    # --- 類別常數：各資料類型的過期天數 ---
    # 對標 C++：static const std::map<std::string, int>
    EXPIRY_DAYS = {
        "stock_price": 1,           # 日股價：FinMind 每天 17:30 更新
        "financial_statement": 90,  # 財報：每季更新一次
        "monthly_revenue": 30,      # 月營收：每月 10 號前公布
    }
    
    def __init__(self, cache_dir: str = "data"):
        """
        建構子
        
        Args:
            cache_dir: 快取根目錄，預設為 "data"
        
        對標 C++：
            CacheManager(const std::string& cache_dir = "data") 
                : cache_dir_(cache_dir) {
                std::filesystem::create_directories(cache_dir_);
            }
        """
        # Path 是路徑物件，支援 / 運算子串接路徑
        # 對標 C++：std::filesystem::path
        self.cache_dir = Path(cache_dir)
        
        # 建立快取目錄（如果不存在）
        # parents=True：自動建立父目錄
        # exist_ok=True：目錄已存在不報錯
        self.cache_dir.mkdir(parents=True, exist_ok=True)
    
    def _generate_key(self, stock_id: str, data_type: str, 
                      start_date: str, end_date: str) -> Path:
        """
        生成快取檔案路徑（私有方法）
        
        Args:
            stock_id: 股票代號，如 "2330"
            data_type: 資料類型，如 "financial_statement"
            start_date: 起始日期，如 "2023-01-01"
            end_date: 結束日期，如 "2024-01-01"
        
        Returns:
            完整檔案路徑，如 data/financial_statement/2330_2023-01-01_2024-01-01.csv
        
        為什麼 key 要包含所有參數？
            不同查詢條件 = 不同資料內容 = 不同快取檔案
            避免 start_date 不同但讀到同一個快取的錯誤
        """
        # 建立子資料夾（依資料類型分類）
        # self.cache_dir / data_type 等於 "data" / "financial_statement"
        type_dir = self.cache_dir / data_type
        type_dir.mkdir(exist_ok=True)
        
        # 組合檔名：股票代號_起始日期_結束日期.csv
        filename = f"{stock_id}_{start_date}_{end_date}.csv"
        
        # 回傳完整路徑
        return type_dir / filename
    
    def _is_expired(self, file_path: Path, data_type: str) -> bool:
        """
        檢查快取是否過期（私有方法）
        
        Args:
            file_path: 快取檔案路徑
            data_type: 資料類型（用來查詢過期天數）
        
        Returns:
            True = 已過期，False = 未過期
        
        對標 C++：
            auto mtime = std::filesystem::last_write_time(path);
            auto now = std::chrono::system_clock::now();
            return (now - mtime) > expiry_days * 24h;
        """
        # stat() 取得檔案狀態，st_mtime 是最後修改時間（Unix timestamp）
        # fromtimestamp() 把 timestamp 轉成 datetime 物件
        mtime = datetime.fromtimestamp(file_path.stat().st_mtime)
        
        # 計算距今幾天
        days_passed = (datetime.now() - mtime).days
        
        # 查詢該資料類型的過期天數，預設 1 天
        # dict.get(key, default)：找不到 key 時回傳 default
        expiry = self.EXPIRY_DAYS.get(data_type, 1)
        
        return days_passed > expiry
    
    def get(self, stock_id: str, data_type: str,
            start_date: str, end_date: str) -> pd.DataFrame | None:
        """
        讀取快取
        
        Args:
            stock_id: 股票代號
            data_type: 資料類型
            start_date: 起始日期
            end_date: 結束日期
        
        Returns:
            DataFrame：快取存在且未過期
            None：快取不存在或已過期
        
        對標 C++：
            std::optional<DataFrame> get(...) {
                if (!exists || expired) return std::nullopt;
                return loadCSV(path);
            }
        """
        file_path = self._generate_key(stock_id, data_type, start_date, end_date)
        
        # 檢查 1：檔案是否存在
        if not file_path.exists():
            print(f"[Cache] 快取不存在：{file_path}")
            return None
        
        # 檢查 2：是否過期
        if self._is_expired(file_path, data_type):
            print(f"[Cache] 快取已過期：{file_path}")
            return None
        
        # 快取命中，讀取並回傳
        print(f"[Cache] 命中快取：{file_path}")
        return pd.read_csv(file_path)
    
    def set(self, stock_id: str, data_type: str,
            start_date: str, end_date: str, df: pd.DataFrame) -> None:
        """
        寫入快取
        
        Args:
            stock_id: 股票代號
            data_type: 資料類型
            start_date: 起始日期
            end_date: 結束日期
            df: 要儲存的 DataFrame
        """
        file_path = self._generate_key(stock_id, data_type, start_date, end_date)
        
        # 儲存為 CSV
        # index=False：不儲存 pandas 自動生成的列索引（0, 1, 2...）
        df.to_csv(file_path, index=False)
        print(f"[Cache] 已寫入快取：{file_path}")


# --- 測試 CacheManager ---
cache = CacheManager("data")
print("CacheManager 初始化成功")
print(f"快取目錄：{cache.cache_dir}")
print(f"過期規則：{cache.EXPIRY_DAYS}")

CacheManager 初始化成功
快取目錄：data
過期規則：{'stock_price': 1, 'financial_statement': 90, 'monthly_revenue': 30}


In [3]:
# =============================================================================
# DataFetcher：資料獲取器
# =============================================================================
# 功能：負責呼叫 FinMind API 獲取各類股票資料
# 職責單一：只管 API 呼叫，不管快取（快取由 CacheManager 負責）
# =============================================================================

from FinMind.data import DataLoader
import pandas as pd
import os

class DataFetcher:
    """
    資料獲取器
    斯41
    職責：
    1. 管理 FinMind API Token
    2. 呼叫各種 FinMind API
    3. 錯誤處理（API 失敗時回傳空 DataFrame）
    
    對標 C++：
    - 類似一個封裝好的 HTTP Client
    - 每個 public method 對應一種 API endpoint
    """
    
    def __init__(self, token: str = None):
        """
        建構子
        
        Args:
            token: FinMind API Token，若不傳則從環境變數讀取
        
        對標 C++：
            DataFetcher(const std::string& token = "") {
                if (token.empty()) {
                    token_ = std::getenv("FINMIND_TOKEN");
                }
            }
        """
        # 如果沒傳 token，從環境變數讀取
        # 這樣 token 不會寫死在程式碼裡（安全性考量）
        self.token = token or os.getenv("FINMIND_TOKEN")
        
        # 初始化 FinMind DataLoader
        self.loader = DataLoader()
        
        # 如果有 token，設定給 loader（提高 API 額度）
        if self.token:
            self.loader.login_by_token(api_token=self.token)
            print("[DataFetcher] 已使用 Token 登入（600次/小時）")
        else:
            print("[DataFetcher] 未設定 Token，使用匿名模式（300次/小時）")
    
    def get_financial_statement(self, stock_id: str, 
                                 start_date: str, end_date: str) -> pd.DataFrame:
        """
        獲取財務報表（綜合損益表）
        
        Args:
            stock_id: 股票代號，如 "2330"
            start_date: 起始日期，如 "2023-01-01"
            end_date: 結束日期，如 "2024-01-01"
        
        Returns:
            DataFrame：成功時回傳財報資料
            空 DataFrame：失敗時回傳（不是 None，方便後續用 .empty 檢查）
        
        為什麼用 try-except？
            API 可能因為網路問題、Token 過期、股票代號錯誤等原因失敗
            用 try-except 確保程式不會崩潰
        
        對標 C++：
            try { ... } catch (const std::exception& e) { ... }
        """
        try:
            print(f"[DataFetcher] 正在獲取 {stock_id} 的財報資料...")
            
            # 呼叫 FinMind API
            df = self.loader.taiwan_stock_financial_statement(
                stock_id=stock_id,
                start_date=start_date,
                end_date=end_date
            )
            
            # 檢查是否有資料
            if df.empty:
                print(f"[DataFetcher] 警告：{stock_id} 財報查無資料")
            else:
                print(f"[DataFetcher] 成功獲取 {len(df)} 筆財報資料")
            
            return df
            
        except Exception as e:
            # 印出錯誤訊息，但不讓程式崩潰
            print(f"[DataFetcher] 錯誤：獲取財報失敗 - {e}")
            # 回傳空 DataFrame（不是 None）
            return pd.DataFrame()
    
    def get_stock_price(self, stock_id: str, 
                        start_date: str, end_date: str) -> pd.DataFrame:
        """
        獲取股價資料（日K線）
        
        Args:
            stock_id: 股票代號
            start_date: 起始日期
            end_date: 結束日期
        
        Returns:
            DataFrame：股價資料（日期、開高低收、成交量等）
        """
        try:
            print(f"[DataFetcher] 正在獲取 {stock_id} 的股價資料...")
            
            df = self.loader.taiwan_stock_daily(
                stock_id=stock_id,
                start_date=start_date,
                end_date=end_date
            )
            
            if df.empty:
                print(f"[DataFetcher] 警告：{stock_id} 股價查無資料")
            else:
                print(f"[DataFetcher] 成功獲取 {len(df)} 筆股價資料")
            
            return df
            
        except Exception as e:
            print(f"[DataFetcher] 錯誤：獲取股價失敗 - {e}")
            return pd.DataFrame()
    
    def get_monthly_revenue(self, stock_id: str, 
                            start_date: str, end_date: str) -> pd.DataFrame:
        """
        獲取月營收資料
        
        Args:
            stock_id: 股票代號
            start_date: 起始日期
            end_date: 結束日期
        
        Returns:
            DataFrame：月營收資料
        """
        try:
            print(f"[DataFetcher] 正在獲取 {stock_id} 的月營收資料...")
            
            df = self.loader.taiwan_stock_month_revenue(
                stock_id=stock_id,
                start_date=start_date,
                end_date=end_date
            )
            
            if df.empty:
                print(f"[DataFetcher] 警告：{stock_id} 月營收查無資料")
            else:
                print(f"[DataFetcher] 成功獲取 {len(df)} 筆月營收資料")
            
            return df
            
        except Exception as e:
            print(f"[DataFetcher] 錯誤：獲取月營收失敗 - {e}")
            return pd.DataFrame()


# --- 測試 DataFetcher ---
fetcher = DataFetcher()
print(f"Token 狀態：{'已設定' if fetcher.token else '未設定'}")

2026-01-22 17:57:11.381 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-01-22 17:57:11.487 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


[DataFetcher] 已使用 Token 登入（600次/小時）
Token 狀態：已設定


In [4]:
# =============================================================================
# DataService：資料服務層（整合 CacheManager + DataFetcher）
# =============================================================================
# 功能：統一的資料獲取介面，自動處理快取邏輯
# 設計模式：Facade Pattern（外觀模式）- 把複雜的子系統包裝成簡單的介面
# =============================================================================

class DataService:
    """
    資料服務層
    
    職責：
    1. 整合 CacheManager 和 DataFetcher
    2. 提供統一的 get_data() 介面
    3. 自動處理「先查快取 → 沒有才打 API → 存入快取」的流程
    
    對標 C++：
    - Facade Pattern：把多個 class 的操作包裝成單一介面
    - 類似 Service Layer 的概念
    """

    import pandas as pd

    
    def __init__(self, cache_dir: str = "data", token: str = None):
        """
        建構子
        
        Args:
            cache_dir: 快取目錄
            token: FinMind API Token
        """
        # 初始化子系統
        self.cache = CacheManager(cache_dir)
        self.fetcher = DataFetcher(token)
        
        # 建立「資料類型 → API 方法」的對應表
        # dict 的 value 是函式本身（不是呼叫結果）
        # 對標 C++：std::map<std::string, std::function<...>>
        self._fetch_methods = {
            "financial_statement": self.fetcher.get_financial_statement,
            "stock_price": self.fetcher.get_stock_price,
            "monthly_revenue": self.fetcher.get_monthly_revenue,
        }
    
    def get_data(self, stock_id: str, data_type: str,
                 start_date: str, end_date: str) -> pd.DataFrame:
        """
        統一的資料獲取介面
        
        流程：
        1. 查快取 → 有且未過期 → 直接回傳
        2. 快取沒有或過期 → 打 API
        3. API 成功 → 存入快取 → 回傳
        4. API 失敗 → 回傳空 DataFrame
        
        Args:
            stock_id: 股票代號，如 "2330"
            data_type: 資料類型，如 "financial_statement", "stock_price", "monthly_revenue"
            start_date: 起始日期，如 "2023-01-01"
            end_date: 結束日期，如 "2024-01-01"
        
        Returns:
            DataFrame：股票資料
        
        使用範例：
            service = DataService()
            df = service.get_data("2330", "financial_statement", "2023-01-01", "2024-01-01")
        """
        # --- Step 1: 查快取 ---
        df = self.cache.get(stock_id, data_type, start_date, end_date)
        
        if df is not None:
            # 快取命中，直接回傳
            return df
        
        # --- Step 2: 快取沒有，打 API ---
        # 檢查 data_type 是否有效
        if data_type not in self._fetch_methods:
            print(f"[DataService] 錯誤：不支援的資料類型 '{data_type}'")
            print(f"[DataService] 支援的類型：{list(self._fetch_methods.keys())}")
            return pd.DataFrame()
        
        # 從 dict 取出對應的方法並呼叫
        # self._fetch_methods[data_type] 是一個函式
        # 後面的 (...) 是呼叫這個函式
        fetch_method = self._fetch_methods[data_type]
        df = fetch_method(stock_id, start_date, end_date)
        
        # --- Step 3: 存入快取 ---
        if not df.empty:
            self.cache.set(stock_id, data_type, start_date, end_date, df)
        
        return df
    
    def get_supported_types(self) -> list:
        """
        取得支援的資料類型列表
        
        Returns:
            list：支援的資料類型，如 ["financial_statement", "stock_price", "monthly_revenue"]
        """
        return list(self._fetch_methods.keys())


# --- 測試 DataService ---
service = DataService()
print("DataService 初始化成功")
print(f"支援的資料類型：{service.get_supported_types()}")

2026-01-22 17:57:11.754 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-01-22 17:57:11.831 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


[DataFetcher] 已使用 Token 登入（600次/小時）
DataService 初始化成功
支援的資料類型：['financial_statement', 'stock_price', 'monthly_revenue']


In [5]:
# =============================================================================
# 測試 DataService：完整的資料獲取流程
# =============================================================================

# 測試 1：獲取財報資料（第一次會打 API，之後會讀快取）
print("=" * 50)
print("測試 1：獲取 2330 財報")
print("=" * 50)
df_financial = service.get_data(
    stock_id="2330",
    data_type="financial_statement",
    start_date="2023-01-01",
    end_date="2024-01-01"
)
print(f"獲取 {len(df_financial)} 筆資料")
print(df_financial.head())

# 測試 2：再次獲取同樣的資料（應該會命中快取）
print("\n" + "=" * 50)
print("測試 2：再次獲取 2330 財報（應命中快取）")
print("=" * 50)
df_financial_cached = service.get_data(
    stock_id="2330",
    data_type="financial_statement",
    start_date="2023-01-01",
    end_date="2024-01-01"
)
print(f"獲取 {len(df_financial_cached)} 筆資料")

測試 1：獲取 2330 財報
[Cache] 命中快取：data\financial_statement\2330_2023-01-01_2024-01-01.csv
獲取 68 筆資料
         date  stock_id                            type         value  \
0  2023-03-31      2330               OperatingExpenses  5.530934e+10   
1  2023-03-31      2330                             TAX  3.732590e+10   
2  2023-03-31      2330                             EPS  7.980000e+00   
3  2023-03-31      2330                    PreTaxIncome  2.442749e+11   
4  2023-03-31      2330  IncomeFromContinuingOperations  2.069490e+11   

      origin_name  
0            營業費用  
1       所得稅費用（利益）  
2       基本每股盈餘（元）  
3        稅前淨利（淨損）  
4  繼續營業單位本期淨利（淨損）  

測試 2：再次獲取 2330 財報（應命中快取）
[Cache] 命中快取：data\financial_statement\2330_2023-01-01_2024-01-01.csv
獲取 68 筆資料


In [6]:
# =============================================================================
# TextProcessor：文字處理器
# =============================================================================
# 功能：將 DataFrame 轉換成自然語言文字片段，供 RAG 使用
# =============================================================================

class TextProcessor:
    """
    文字處理器
    
    職責：
    1. 將財報 DataFrame 轉換成自然語言句子
    2. 日期轉換成季度格式
    3. 數字格式化（大數字轉成億元）
    """
    
    # --- 指標名稱對照表（英文 → 中文）---
    # 只處理這些重要指標
    INDICATOR_NAMES = {
        "EPS": "每股盈餘(EPS)",
        "Revenue": "營收",
        "GrossProfit": "毛利",
        "OperatingIncome": "營業利益",
        "PreTaxIncome": "稅前淨利",
        "NetIncome": "淨利",
        "OperatingExpenses": "營業費用",
    }
    
    # --- 月份 → 季度對照表 ---
    # 財報日期固定是季末：3月底、6月底、9月底、12月底
    MONTH_TO_QUARTER = {
        3: 1,   # 03-31 → Q1
        6: 2,   # 06-30 → Q2
        9: 3,   # 09-30 → Q3
        12: 4,  # 12-31 → Q4
    }
    
    def _date_to_quarter(self, date_str: str) -> str:
        """
        將日期字串轉換成季度格式
        
        Args:
            date_str: 日期字串，如 "2023-03-31"
        
        Returns:
            季度字串，如 "2023年第1季"
        
        範例：
            "2023-03-31" → "2023年第1季"
            "2023-06-30" → "2023年第2季"
        """
        # 拆解日期字串
        # "2023-03-31".split("-") → ["2023", "03", "31"]
        parts = date_str.split("-")
        year = parts[0]
        month = int(parts[1])  # "03" → 3
        
        # 查表得到季度
        quarter = self.MONTH_TO_QUARTER.get(month, 1)
        
        return f"{year}年第{quarter}季"
    
    def _format_number(self, value: float, indicator_type: str) -> str:
        """
        格式化數字，讓大數字更易讀
        
        Args:
            value: 數值
            indicator_type: 指標類型（用來判斷單位）
        
        Returns:
            格式化後的字串
        
        範例：
            508632973000, "Revenue" → "5,086.33 億元"
            7.98, "EPS" → "7.98 元"
        """
        # EPS 是「每股」金額，通常是個位數到十位數，直接顯示
        if indicator_type == "EPS":
            return f"{value:.2f} 元"
        
        # 其他指標（營收、淨利等）通常是很大的數字
        # 轉換成「億元」比較好讀
        if abs(value) >= 1e8:  # 大於等於 1 億
            value_in_billion = value / 1e8  # 除以 1 億
            return f"{value_in_billion:,.2f} 億元"
        elif abs(value) >= 1e4:  # 大於等於 1 萬
            value_in_ten_thousand = value / 1e4
            return f"{value_in_ten_thousand:,.2f} 萬元"
        else:
            return f"{value:,.2f} 元"
    
    def df_to_chunks(self, df: pd.DataFrame, stock_name: str = None) -> list[str]:
        """
        將 DataFrame 轉換成文字片段列表
        
        Args:
            df: 財報 DataFrame，需包含 date, stock_id, type, value 欄位
            stock_name: 股票名稱（可選），如 "台積電"
        
        Returns:
            文字片段列表
        
        範例輸出：
            ["2023年第1季 台積電(2330) 的每股盈餘(EPS)為 7.98 元。",
             "2023年第1季 台積電(2330) 的營收為 5,086.33 億元。"]
        """
        chunks = []
        
        # 篩選我們關心的指標
        # df['type'].isin([...]) 會回傳 True/False 的 Series
        # df[...] 會篩選出 True 的那些 row
        target_types = list(self.INDICATOR_NAMES.keys())
        filtered_df = df[df['type'].isin(target_types)]
        
        # 遍歷每一筆資料
        # iterrows() 會回傳 (index, row) 的 tuple
        # _ 表示我們不需要 index，只要 row
        for _, row in filtered_df.iterrows():
            # 取得欄位值
            date_str = row['date']
            stock_id = row['stock_id']
            indicator_type = row['type']
            value = row['value']
            
            # 轉換格式
            quarter_str = self._date_to_quarter(date_str)
            indicator_name = self.INDICATOR_NAMES[indicator_type]
            value_str = self._format_number(value, indicator_type)
            
            # 組合股票名稱
            if stock_name:
                stock_str = f"{stock_name}({stock_id})"
            else:
                stock_str = f"股票{stock_id}"
            
            # 組合成句子
            chunk = f"{quarter_str} {stock_str} 的{indicator_name}為 {value_str}。"
            chunks.append(chunk)
        
        return chunks


# --- 測試 TextProcessor ---
processor = TextProcessor()

# 用之前取得的資料測試
test_chunks = processor.df_to_chunks(df_financial, stock_name="台積電")

print(f"共產生 {len(test_chunks)} 個文字片段")
print("\n前 5 個片段：")
for i, chunk in enumerate(test_chunks[:5]):
    print(f"{i+1}. {chunk}")

共產生 24 個文字片段

前 5 個片段：
1. 2023年第1季 台積電(2330) 的營業費用為 553.09 億元。
2. 2023年第1季 台積電(2330) 的每股盈餘(EPS)為 7.98 元。
3. 2023年第1季 台積電(2330) 的稅前淨利為 2,442.75 億元。
4. 2023年第1季 台積電(2330) 的營收為 5,086.33 億元。
5. 2023年第1季 台積電(2330) 的營業利益為 2,312.38 億元。


In [7]:
# =============================================================================
# EmbeddingService：向量化服務（你自己寫的！）
# =============================================================================
# 功能：將文字轉換成向量（embedding），使用 Ollama API
# =============================================================================

import requests

class EmbeddingService:
    """
    向量化服務
    
    職責：
    1. 連接 Ollama API
    2. 將文字轉換成向量
    """

    def __init__(self, model="nomic-embed-text", base_url="http://localhost:11434"):
        """
        建構子
        
        Args:
            model: 模型名稱，預設 nomic-embed-text
            base_url: Ollama API 網址，預設 localhost
        """
        self.model = model 
        self.base_url = base_url

    def embed(self, text: str) -> list[float]:
        """
        將文字轉換成向量
        
        Args:
            text: 要轉換的文字
        
        Returns:
            向量（list of floats）
        """
        URL = f"{self.base_url}/api/embeddings"
        model = self.model
        prompt = text
        response = requests.post(
            URL, 
            json = {
                "model": model,
                "prompt": prompt
            }
        )
        result = response.json()
        vector = result["embedding"]
        return vector


# --- 測試 EmbeddingService ---
embedding_service = EmbeddingService()
print("EmbeddingService 初始化成功")
print(f"模型：{embedding_service.model}")
print(f"API 網址：{embedding_service.base_url}")

# 測試向量化
test_text = "2023年第1季 台積電(2330) 的每股盈餘(EPS)為 7.98 元。"
print(f"\n測試文字：{test_text}")

vector = embedding_service.embed(test_text)
print(f"向量維度：{len(vector)}")
print(f"向量前 5 個值：{vector[:5]}")

EmbeddingService 初始化成功
模型：nomic-embed-text
API 網址：http://localhost:11434

測試文字：2023年第1季 台積電(2330) 的每股盈餘(EPS)為 7.98 元。
向量維度：768
向量前 5 個值：[-0.7738291621208191, 0.7138004899024963, -3.4446558952331543, -0.3421982228755951, 0.9335871934890747]


In [ ]:
# =============================================================================
# VectorStore：向量資料庫（你自己寫的！）
# =============================================================================
# 功能：使用 ChromaDB 儲存和查詢向量
# =============================================================================

import chromadb

class VectorStore:
    """
    向量資料庫
    
    職責：
    1. 連接 ChromaDB（持久化模式）
    2. 將文字 chunks + 向量存入資料庫
    3. 根據查詢向量，找出最相似的 chunks
    """
    
    def __init__(self, persist_path: str, collection_name: str = "defaultCollection",
                 embedding_service: EmbeddingService = None):
        """
        建構子
        
        Args:
            persist_path: 資料庫儲存路徑
            collection_name: Collection 名稱，預設 "defaultCollection"
            embedding_service: EmbeddingService 實例，用來產生向量
        """
        self.client = chromadb.PersistentClient(path=persist_path)
        self.collection = self.client.get_or_create_collection(name=collection_name)
        self.embedding_service = embedding_service

    def add(self, chunks: list[str], stock_id: str):
        """
        將多個文字 chunks 存入 ChromaDB
        
        Args:
            chunks: 文字列表
            stock_id: 股票代號（存在 metadata 中）
        """
        ids = []
        embeddings = []
        metadatas = []
        
        for i, chunk in enumerate(chunks):
            # 產生唯一 ID
            id = f"{stock_id}_{i}"
            ids.append(id)
            
            # 產生向量
            embed = self.embedding_service.embed(chunk)
            embeddings.append(embed)
            
            # 準備 metadata
            metadata = {"stock_id": stock_id}
            metadatas.append(metadata)

        # 存入 ChromaDB
        self.collection.add(
            ids=ids,
            documents=chunks,
            embeddings=embeddings,
            metadatas=metadatas
        )
        print(f"[VectorStore] 已存入 {len(chunks)} 個 chunks，stock_id={stock_id}")

    def query(self, text: str, n_results: int = 3, stock_id: str = None):
        """
        搜尋最相似的 chunks
        
        Args:
            text: 查詢文字
            n_results: 回傳幾筆結果，預設 3
            stock_id: 可選，只搜尋特定股票
        
        Returns:
            找到的文字列表（二維 list）
        """
        # 把查詢文字轉成向量
        embedding = self.embedding_service.embed(text)
        
        # 查詢 ChromaDB
        if stock_id:
            result = self.collection.query(
                query_embeddings=[embedding],
                n_results=n_results,
                where={"stock_id": stock_id}
            )
        else:
            result = self.collection.query(
                query_embeddings=[embedding],
                n_results=n_results,
            )

        return result["documents"]


# --- 測試 VectorStore ---
# 初始化（需要先執行 EmbeddingService cell）
vector_store = VectorStore(
    persist_path="./chroma_db",
    collection_name="financial_reports",
    embedding_service=embedding_service
)
print("VectorStore 初始化成功")
print(f"Collection 名稱：{vector_store.collection.name}")

# 存入測試資料（用之前產生的 chunks）
print(f"\n準備存入 {len(test_chunks)} 個 chunks...")
vector_store.add(chunks=test_chunks, stock_id="2330")

# 測試查詢
print("\n--- 測試查詢 ---")
query_text = "台積電的 EPS 是多少？"
print(f"查詢：{query_text}")
results = vector_store.query(query_text, n_results=3)
print(f"找到 {len(results[0])} 筆結果：")
for i, doc in enumerate(results[0]):
    print(f"  {i+1}. {doc}")

VectorStore 初始化成功
Collection 名稱：financial_reports

準備存入 24 個 chunks...
[VectorStore] 已存入 24 個 chunks，stock_id=2330

--- 測試查詢 ---
查詢：台積電的 EPS 是多少？
找到 3 筆結果：
  1. 2023年第4季 台積電(2330) 的營收為 6,255.29 億元。
  2. 2023年第2季 台積電(2330) 的每股盈餘(EPS)為 7.01 元。
  3. 2023年第4季 台積電(2330) 的每股盈餘(EPS)為 9.21 元。


TypeError: list indices must be integers or slices, not str

In [ ]:
# =============================================================================
# RAGService：RAG 服務（你自己寫的！）
# =============================================================================
# 功能：整合檢索（Retrieval）和生成（Generation），回答使用者問題
# =============================================================================

import requests

class RAGService:
    """
    RAG 服務
    
    職責：
    1. 收到使用者問題
    2. 用 VectorStore 找相關 chunks
    3. 組成 prompt（問題 + chunks）
    4. 呼叫 Ollama LLM 生成回答
    """
    
    def __init__(self, vector_store: VectorStore, 
                 model: str = "gemma3:12b", base_url: str = "http://localhost:11434"):
        """
        建構子
        
        Args:
            vector_store: VectorStore 實例，用來搜尋相關 chunks
            model: LLM 模型名稱，預設 gemma3:12b
            base_url: Ollama API 網址
        """
        self.vector_store = vector_store
        self.model = model
        self.base_url = base_url
    
    def ask(self, question: str) -> str:
        """
        回答使用者問題
        
        Args:
            question: 使用者的問題
        
        Returns:
            LLM 生成的回答
        """
        # 1. 搜尋相關 chunks
        chunks = self.vector_store.query(question)
        
        # 2. 組成 prompt
        context = "\n".join(chunks[0])
        prompt = f"""根據以下資料回答問題：

{context}

問題：{question}
"""
        
        # 3. 呼叫 Ollama API
        response = requests.post(
            f"{self.base_url}/api/generate",
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            }
        )
        result = response.json()
        answer = result["response"]
        
        # 4. 回傳答案
        return answer


# --- 測試 RAGService ---
# 初始化（需要先執行 VectorStore cell）
rag = RAGService(
    vector_store=vector_store,
    model="gemma3:12b"  # 確保你有安裝這個模型：ollama pull gemma3:12b
)
print("RAGService 初始化成功")
print(f"使用模型：{rag.model}")

# 測試問答
print("\n" + "=" * 50)
print("測試 RAG 問答")
print("=" * 50)

question = "台積電 2023 年的 EPS 表現如何？"
print(f"問題：{question}")
print("\n回答：")
answer = rag.ask(question)
print(answer)

RAGService 初始化成功
使用模型：gemma3:12b

測試 RAG 問答
問題：台積電 2023 年的 EPS 表現如何？

回答：
根據資料，台積電 2023 年第一季的每股盈餘 (EPS) 為 **7.98 元**。

因此，台積電 2023 年的第一季 EPS 表現為 7.98 元。



In [9]:
def transform_to_sentences(df):
    """
    將 DataFrame 的數字轉化為 AI 好讀的自然語言
    """
    sentences = []
    # 篩選我們感興趣的指標 (例如：營收、淨利)
    target_indicators = ['Revenue', 'Net_Income_Loss', 'EPS']
    filtered_df = df[df['type'].isin(target_indicators)]

    for _, row in filtered_df.iterrows():
        sentence = f"{row['date']} {row['stock_id']} 的 {row['type']} 為 {row['value']} 元。"
        sentences.append(sentence)
    
    return sentences

# 轉化並列印前五條
report_sentences = transform_to_sentences(df_2330)
for s in report_sentences[:5]:
    print(f"生成文本: {s}")

生成文本: 2023-03-31 2330 的 EPS 為 7.98 元。
生成文本: 2023-03-31 2330 的 Revenue 為 508632973000.0 元。
生成文本: 2023-06-30 2330 的 EPS 為 7.01 元。
生成文本: 2023-06-30 2330 的 Revenue 為 480841254000.0 元。
生成文本: 2023-09-30 2330 的 Revenue 為 546732758000.0 元。
